# SK-RD4AD — Data Augmentation Campaign (Colab, resumable batch)
Setup and dataset copy run **once**, then a loop iterates over a list of configs (baseline + ablations). Each run gets its own folder on Drive.

**Resumable:** if you re-run the notebook, runs already completed (marked with a `DONE` file) are **skipped** — the baseline is not recomputed. Add configs to the list anytime: only new ones are executed. A failure in one run does not stop the others.


In [1]:
from google.colab import drive
import os, shutil, glob, json, subprocess, time
from datetime import datetime

drive.mount('/content/drive')

BRANCH = "feature/ablatable-augmentation"
if not os.path.isdir('/content/sk-rd4ad'):
    !git clone --branch {BRANCH} https://github.com/emanuelepietrocometti/sk-rd4ad.git
%cd /content/sk-rd4ad

!pip install -q --extra-index-url https://download.pytorch.org/whl/cu130 \
    torch>=2.1.0 torchvision>=0.16.0 numpy pandas scipy imageio matplotlib \
    opencv-python opencv-contrib-python Pillow scikit-image scikit-learn \
    fastprogress geomloss tqdm


Mounted at /content/drive
Cloning into 'sk-rd4ad'...
remote: Enumerating objects: 220, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 220 (delta 75), reused 75 (delta 51), pack-reused 116 (from 1)
Receiving objects: 100% (220/220), 139.14 KiB | 13.91 MiB/s, done.
Resolving deltas: 100% (125/125), done.
/content/sk-rd4ad


# Campaign configuration
`HPARAMS` = the optimized configuration from ch. 5, **fixed** for the whole campaign. `AUG_CONFIGS` = the list of augmentation files to try; `SEEDS` = the list of seeds. Runs are the product `AUG_CONFIGS x SEEDS`, each in its own folder.


In [11]:
# === DATASET / DRIVE ===
CLASS_NAME = "reda_dustValidationAndTrain"          # custom textile class (ch. 5 dataset with noise in training)
DRIVE_BASE_PATH    = "/content/drive/MyDrive/Tesi/experiments/results_SKRD4AD_augmentation"
DRIVE_DATASET_PATH = f"/content/drive/MyDrive/Tesi/datasets/MVTec/{CLASS_NAME}"
LOCAL_DATASET_PATH = f"./mvtec/{CLASS_NAME}"

# === RUN LIST ===
# Add/remove configs freely: already-completed runs are skipped on re-run.
AUG_CONFIGS = [
    "configs/aug_off.json",
    "configs/aug_legacy.json",
    "configs/oat_affine.json",
    "configs/oat_blur.json",
    "configs/oat_brightness_contrast.json",
    "configs/oat_dynamic_crop.json",
    "configs/oat_equalize.json",
    "configs/oat_grayscale.json",
    "configs/oat_hflip.json",
    "configs/oat_hue.json",
    "configs/oat_speckle_high.json",
    "configs/oat_speckle_low.json",
    "configs/oat_vflip.json",
    "configs/combined_geometric.json",
    "configs/combined_candidate.json",
]
SEEDS = [42]   # fixed seed for the whole campaign

# === HPARAMS: OPTIMIZED configuration from ch. 5 (Table 4.2.4), FIXED ===
HPARAMS = {
    "learning_rate": 0.008783859851350453, "batch_size": 64, "res": 2,
    "layerloss": 0, "rate": 0.05, "L2": 0, "net": "res18",
    "seg": 1, "vis": 0, "print_epoch": 50, "epochs": 200,
}

# Guard: config files must be present on the branch
missing = [c for c in AUG_CONFIGS if not os.path.exists(c)]
if missing:
    raise FileNotFoundError(f"Configs not found (did you push configs/ to branch '{BRANCH}'?): {missing}")

RUNS = [(cfg, s) for cfg in AUG_CONFIGS for s in SEEDS]
print(f"{len(RUNS)} runs queued ({len(AUG_CONFIGS)} configs x {len(SEEDS)} seeds)")


15 runs queued (15 configs x 1 seeds)


# Dataset preparation (once)
Copy the dataset to fast local storage. If already present in this session, skip.


In [3]:
if not (os.path.isdir(LOCAL_DATASET_PATH) and os.listdir(LOCAL_DATASET_PATH)):
    print(f"Copying {CLASS_NAME} from Drive to local storage...")
    if os.path.exists(LOCAL_DATASET_PATH):
        shutil.rmtree(LOCAL_DATASET_PATH)
    shutil.copytree(DRIVE_DATASET_PATH, LOCAL_DATASET_PATH)
    print("Dataset copied.")
else:
    print("Dataset already present locally, skipping copy.")
os.makedirs(DRIVE_BASE_PATH, exist_ok=True)


Copying reda_dustValidationAndTrain from Drive to local storage...
Dataset copied.


# Resumable batch runner
For each `(config, seed)`: skip if `DONE` exists; otherwise train, evaluate and sync to Drive, writing the `DONE` marker at the end of the run. Partial (interrupted) runs are cleaned up and redone. A failure in one run does not stop the others.


In [12]:
LOCAL_CKPT, LOCAL_RES = "./checkpoints", "./results"

def _clean(*dirs):
    for d in dirs:
        if os.path.exists(d): shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)

def _sync(local_dir, drive_dir):
    if not os.path.isdir(local_dir): return
    os.makedirs(drive_dir, exist_ok=True)
    shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)

def run_one(aug_config, seed):
    subprocess.run("rm -f /dev/shm/torch_* 2>/dev/null", shell=True)
    tag      = os.path.splitext(os.path.basename(aug_config))[0]
    run_name = f"aug_{tag}_{CLASS_NAME}_seed{seed}"
    proj     = f"skrd4ad_{run_name}"
    drun     = os.path.join(DRIVE_BASE_PATH, run_name)
    dckpt, dres = os.path.join(drun, "checkpoints"), os.path.join(drun, "results")
    done_marker = os.path.join(drun, "DONE")

    if os.path.exists(done_marker):
        print(f"[skip] {run_name} (already completed)"); return "skipped"
    # partial/interrupted run -> clean up and redo
    if os.path.exists(drun):
        print(f"[redo] {run_name}: incomplete folder, cleaning up")
        shutil.rmtree(drun)

    with open(aug_config) as f: aug_cfg_content = json.load(f)
    _clean(LOCAL_CKPT, LOCAL_RES)
    os.makedirs(dckpt, exist_ok=True); os.makedirs(dres, exist_ok=True)
    with open(os.path.join(drun, "run_config.json"), "w") as f:
        json.dump({"run_name": run_name, "class_": CLASS_NAME, "seed": seed,
                   "project_name": proj, "dataset": DRIVE_DATASET_PATH,
                   "started_at": datetime.now().isoformat(timespec="seconds"),
                   "hparams": HPARAMS, "aug_config_path": aug_config,
                   "aug_config": aug_cfg_content}, f, indent=2)

    print(f"\n===== TRAIN {run_name} =====")
    train_cmd = ["python", "main.py",
        "--class_", CLASS_NAME, "--data_path", "./mvtec/",
        "--ckpt_path", LOCAL_CKPT + "/", "--img_path", LOCAL_RES + "/",
        "--project_name", proj, "--aug-config", aug_config, "--seed", str(seed),
        "--net", HPARAMS["net"], "--L2", str(HPARAMS["L2"]), "--res", str(HPARAMS["res"]),
        "--rate", str(HPARAMS["rate"]), "--batch_size", str(HPARAMS["batch_size"]),
        "--layerloss", str(HPARAMS["layerloss"]), "--seg", str(HPARAMS["seg"]),
        "--vis", str(HPARAMS["vis"]), "--print_epoch", str(HPARAMS["print_epoch"]),
        "--learning_rate", str(HPARAMS["learning_rate"]), "--epochs", str(HPARAMS["epochs"]),
        "--cut", str(0)]

    res = subprocess.run(train_cmd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    print(res.stdout[-4000:])                      # tail live
    if res.returncode != 0:
        fdir = os.path.join(DRIVE_BASE_PATH, "_failures"); os.makedirs(fdir, exist_ok=True)
        open(os.path.join(fdir, run_name + ".log"), "w").write(res.stdout)
        raise RuntimeError(f"training failed (rc={res.returncode}); log in _failures/")

    ckpts = glob.glob(os.path.join(LOCAL_CKPT, "*.pth"))
    if not ckpts:
        raise FileNotFoundError("no checkpoint produced")
    latest = max(ckpts, key=os.path.getmtime)
    eval_img = os.path.join(LOCAL_RES, "eval_report"); os.makedirs(eval_img, exist_ok=True)
    print(f"===== EVAL {run_name} ({os.path.basename(latest)}) =====")
    eval_cmd = ["python", "eval.py",
        "--class_", CLASS_NAME, "--data_path", "./mvtec/",
        "--checkpoint_path", latest, "--img_path", eval_img + "/",
        "--net", HPARAMS["net"], "--res", str(HPARAMS["res"]), "--seg", str(HPARAMS["seg"])]
    res_eval = subprocess.run(eval_cmd, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True)
    print(res_eval.stdout[-4000:])
    if res_eval.returncode != 0:
        fdir = os.path.join(DRIVE_BASE_PATH, "_failures"); os.makedirs(fdir, exist_ok=True)
        open(os.path.join(fdir, run_name + "_eval.log"), "w").write(res_eval.stdout)
        raise RuntimeError(f"eval failed (rc={res_eval.returncode}); log in _failures/")

    _sync(LOCAL_CKPT, dckpt); _sync(LOCAL_RES, dres)
    cfgp = os.path.join(drun, "run_config.json")
    c = json.load(open(cfgp)); c["finished_at"] = datetime.now().isoformat(timespec="seconds")
    json.dump(c, open(cfgp, "w"), indent=2)
    open(done_marker, "w").write(datetime.now().isoformat())
    print(f"[ok] {run_name} synced to Drive")
    return "done"

summary = {"done": [], "skipped": [], "failed": []}
for i, (aug_config, seed) in enumerate(RUNS, 1):
    print(f"\n########## RUN {i}/{len(RUNS)} ##########")
    try:
        status = run_one(aug_config, seed)
        summary[status].append((aug_config, seed))
    except Exception as e:
        print(f"[FAIL] {aug_config} seed={seed}: {e}")
        summary["failed"].append((aug_config, seed))

print("\n===== SUMMARY =====")
for k in ("done", "skipped", "failed"):
    print(f"{k}: {len(summary[k])} -> {[f'{os.path.basename(c)}#{s}' for c,s in summary[k]]}")



########## RUN 1/15 ##########

===== TRAIN aug_aug_off_reda_dustValidationAndTrain_seed42 =====
poch [88/200], loss:0.1497
epoch [89/200], loss:0.1496
epoch [90/200], loss:0.1498
epoch [91/200], loss:0.1496
epoch [92/200], loss:0.1495
epoch [93/200], loss:0.1480
epoch [94/200], loss:0.1497
epoch [95/200], loss:0.1494
epoch [96/200], loss:0.1492
epoch [97/200], loss:0.1479
epoch [98/200], loss:0.1489
epoch [99/200], loss:0.1460
epoch [100/200], loss:0.1453
Pixel AUROC: 0.953, Sample AUROC: 0.910, AUPRO: 0.930
AP-loc: 0.562, F1-Score: 0.968, Precision: 0.953, Recall: 0.984, F1-px: 0.582
max_auc =  0.953
max_epoch =  100
max_pr =  0.93
max_epoch =  100
epoch [101/200], loss:0.1465
epoch [102/200], loss:0.1464
epoch [103/200], loss:0.1462
epoch [104/200], loss:0.1460
epoch [105/200], loss:0.1452
epoch [106/200], loss:0.1446
epoch [107/200], loss:0.1482
epoch [108/200], loss:0.1466
epoch [109/200], loss:0.1446
epoch [110/200], loss:0.1448
epoch [111/200], loss:0.1434
epoch [112/200], loss

KeyboardInterrupt: 

# Campaign status
Scan Drive and show which runs are complete and which are missing.


In [ ]:
import os, glob, re, json
import pandas as pd

# === Paths / campaign identity (must match launch_batch_augmentation.ipynb) ===
ABLATION_BASE = "/content/drive/MyDrive/Tesi/experiments/results_SKRD4AD_augmentation"
CLASS_NAME    = "reda_dustValidationAndTrain"
SEEDS         = [42]

# Where the aug configs live. If this dir is missing/empty we fall back to
# discovering run folders directly on Drive (robust for post-hoc aggregation).
CONFIG_DIR = "/content/sk-rd4ad/configs"

# report.txt label  ->  canonical column name (aligned with the SSN summary)
LABEL_MAP = {
    "AUROC":             "I-AUROC",     # sample/image level
    "Pixel AUROC":       "P-AUROC",     # pixel level
    "AUPRO":             "AUPRO",
    "AP-loc":            "AP-loc",
    "F1-Score":          "F1-score",
    "Accuracy":          "Accuracy",
    "Precision":         "Precision",
    "Recall":            "Recall",
    "Optimal Threshold": "Threshold",
}

_num = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def parse_report(path):
    """Parse SK-RD4AD's report.txt into {canonical_name: float}."""
    out = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if ":" not in line:
                continue
            label, value = line.split(":", 1)
            key = LABEL_MAP.get(label.strip())
            if key is None:
                continue
            m = _num.search(value)
            if m:
                out[key] = float(m.group())
    return out

def config_tags():
    """Config basenames (tags) to look for; fall back to Drive discovery."""
    cfgs = sorted(glob.glob(os.path.join(CONFIG_DIR, "*.json")))
    if cfgs:
        return [os.path.splitext(os.path.basename(c))[0] for c in cfgs], False
    # Fallback: derive tags from existing run folders on Drive
    tags = set()
    pat = re.compile(rf"^aug_(.+)_{re.escape(CLASS_NAME)}_seed\d+$")
    for d in sorted(glob.glob(os.path.join(ABLATION_BASE, "aug_*"))):
        m = pat.match(os.path.basename(d))
        if m:
            tags.add(m.group(1))
    return sorted(tags), True

tags, discovered = config_tags()
if discovered:
    print(f"[info] configs/ empty or missing -> discovered {len(tags)} tags from Drive runs")

rows = []
for tag in tags:
    for seed in SEEDS:
        run_name = f"aug_{tag}_{CLASS_NAME}_seed{seed}"
        run_dir  = os.path.join(ABLATION_BASE, run_name)
        row = {"config": tag, "seed": seed}

        # status from the DONE marker written by the batch runner
        row["status"] = "done" if os.path.exists(os.path.join(run_dir, "DONE")) else "not_run"

        reports = glob.glob(os.path.join(run_dir, "**", "report.txt"), recursive=True)
        if not reports:
            if row["status"] == "done":
                row["status"] = "no_report"   # marked done but report missing
            rows.append(row)
            continue

        row.update(parse_report(sorted(reports)[0]))
        rows.append(row)

df = pd.DataFrame(rows)

if not df.empty:
    front = [c for c in ["config", "seed", "status",
                         "I-AUROC", "P-AUROC", "AUPRO", "AP-loc",
                         "F1-score", "Accuracy", "Precision", "Recall", "Threshold"]
             if c in df.columns]
    df = df[front + [c for c in df.columns if c not in front]]
    # Rank done runs by image-level AUROC (best first), keep the rest at the bottom
    if "I-AUROC" in df.columns:
        df = df.sort_values(["status", "I-AUROC"], ascending=[True, False],
                            na_position="last").reset_index(drop=True)

out_csv = os.path.join(ABLATION_BASE, f"ablation_summary_{CLASS_NAME}.csv")
df.to_csv(out_csv, index=False)
print(df.to_string(index=False))
print("\nSaved summary ->", out_csv)

In [ ]:
from google.colab import runtime
runtime.unassign()